In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\shivn\DE-Interview-Prep\.venv\Scripts\python.exe"

if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BTS Flight Delay - Q1 2024") \
    .master("local[*]") \
    .getOrCreate()

print("Spark is running!")
print(f"Spark version: {spark.version}")

Spark is running!
Spark version: 3.5.1


In [5]:
jan = spark.read.csv("../data/raw/January_2024.csv", header=True, inferSchema=True)
feb = spark.read.csv("../data/raw/February_2024.csv", header=True, inferSchema=True)
mar = spark.read.csv("../data/raw/March_2024.csv", header=True, inferSchema=True)

df = jan.unionByName(feb).unionByName(mar)

print(f"January rows:  {jan.count():,}")
print(f"February rows: {feb.count():,}")
print(f"March rows:    {mar.count():,}")
print(f"Total Q1 2024: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

January rows:  547,271
February rows: 519,221
March rows:    591,767
Total Q1 2024: 1,658,259
Total columns: 37


In [6]:
df.show(5)

+----+-----+------------+-----------+--------------------+-----------------+--------+-----------------+------+----------------+----------------+----+--------------+--------------+------------+--------+---------+-------------+---------+------------+--------+---------+-------------+---------+---------+-----------------+--------+----------------+-------------------+--------+-------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|             FL_DATE|OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|DEST|DEST_CITY_NAME|DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+------------+-----------+--------------------+------------

In [7]:
df.show(5,truncate=False,vertical=True)

-RECORD 0-----------------------------------
 YEAR                | 2024                 
 MONTH               | 1                    
 DAY_OF_MONTH        | 1                    
 DAY_OF_WEEK         | 1                    
 FL_DATE             | 1/1/2024 12:00:00 AM 
 OP_UNIQUE_CARRIER   | 9E                   
 TAIL_NUM            | N131EV               
 OP_CARRIER_FL_NUM   | 5225                 
 ORIGIN              | ATL                  
 ORIGIN_CITY_NAME    | Atlanta, GA          
 ORIGIN_STATE_ABR    | GA                   
 DEST                | AVL                  
 DEST_CITY_NAME      | Asheville, NC        
 DEST_STATE_ABR      | NC                   
 CRS_DEP_TIME        | 1410                 
 DEP_TIME            | 1406                 
 DEP_DELAY           | -4.0                 
 DEP_DELAY_NEW       | 0.0                  
 DEP_DEL15           | 0.0                  
 CRS_ARR_TIME        | 1511                 
 ARR_TIME            | 1454                 
 ARR_DELAY

In [5]:
# Quick data health check
print(f"Total rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nNull counts per column:")
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) 
           for c in df.columns]).show(vertical=True)

Total rows: 1,658,259
Total columns: 37

Null counts per column:
-RECORD 0----------------------
 YEAR                | 0       
 MONTH               | 0       
 DAY_OF_MONTH        | 0       
 DAY_OF_WEEK         | 0       
 FL_DATE             | 0       
 OP_UNIQUE_CARRIER   | 0       
 TAIL_NUM            | 6570    
 OP_CARRIER_FL_NUM   | 0       
 ORIGIN              | 0       
 ORIGIN_CITY_NAME    | 0       
 ORIGIN_STATE_ABR    | 0       
 DEST                | 0       
 DEST_CITY_NAME      | 0       
 DEST_STATE_ABR      | 0       
 CRS_DEP_TIME        | 0       
 DEP_TIME            | 27557   
 DEP_DELAY           | 27679   
 DEP_DELAY_NEW       | 27679   
 DEP_DEL15           | 27679   
 CRS_ARR_TIME        | 0       
 ARR_TIME            | 29028   
 ARR_DELAY           | 32207   
 ARR_DELAY_NEW       | 32207   
 ARR_DEL15           | 32207   
 CANCELLED           | 0       
 CANCELLATION_CODE   | 1629748 
 DIVERTED            | 0       
 CRS_ELAPSED_TIME    | 1       
 ACTUAL

In [8]:
# Verify the %  NULL pattern
total_rows = 1_658_259
delay_cause_nulls = 1_327_781
null_percentage = delay_cause_nulls / total_rows * 100
print(f"Delay cause NULL %: {null_percentage:.1f}%")

Delay cause NULL %: 80.1%


In [9]:
# Verify: when ARR_DEL15 = 0, are delay columns always NULL?

df.filter(
    (col("ARR_DEL15") == 0) & 
    (col("CARRIER_DELAY").isNotNull())
).count()

# If this returns 0 → rule confirmed
# If this returns > 0 → rule violated, investigate

0

In [13]:
# PROBLEM 1 — Basic DataFrame Operations
# 1. Print the schema (column names + data types)
# 2. Select only these 5 columns:
#    OP_UNIQUE_CARRIER, ORIGIN, DEST,
#    DEP_DELAY, ARR_DELAY
# 3. Show 5 rows of this smaller DataFrame

df.printSchema()
df.select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY").show(5)

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- A

In [22]:
# PROBLEM 2 — Filtering
# Using your BTS DataFrame:
#
# 1. Filter only DELAYED flights (ARR_DEL15 = 1)
# 2. From those, filter flights where ARR_DELAY > 60
#    (delayed more than 1 hour)
# 3. Select these columns:
#    OP_UNIQUE_CARRIER, ORIGIN, DEST, ARR_DELAY
# 4. Show 10 rows
# 5. Print how many flights were delayed more than 1 hour
#
# Hint: df.filter(col("column") == value)
#       You can chain .filter().filter()
#       OR combine with & operator


delayed = df.filter((col("ARR_DEL15") == 1) & (col("ARR_DELAY") > 60)).select("OP_UNIQUE_CARRIER", "ORIGIN", "DEST", "ARR_DELAY")
delayed.show(10)
print(f"Flights delayed more than 1 hour: {delayed.count():,}")


+-----------------+------+----+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|ARR_DELAY|
+-----------------+------+----+---------+
|               9E|   AVL| ATL|    207.0|
|               9E|   CLT| JFK|    162.0|
|               9E|   RDU| LGA|    204.0|
|               AA|   SAT| CLT|     78.0|
|               AA|   BOS| LAX|     92.0|
|               AA|   AUS| PHX|    108.0|
|               AA|   PHX| PDX|     61.0|
|               AA|   DFW| AUS|    121.0|
|               AA|   PHL| CLT|     83.0|
|               AA|   CLT| PHL|     91.0|
+-----------------+------+----+---------+
only showing top 10 rows

Flights delayed more than 1 hour: 110,406


In [24]:
# PROBLEM 3 — Aggregations
# Using your BTS DataFrame:
#
# 1. Find the TOP 5 carriers by:
#    → total number of flights
#    → average arrival delay (all flights, not just delayed)
#    → total flights delayed more than 15 mins
#
# 2. Sort by total flights descending
#
# 3. Show the result
#
# Hint:
# from pyspark.sql.functions import count, avg, sum, round
# df.groupBy("column").agg(
#     count("*").alias("total_flights"),
#     avg("column").alias("avg_delay")
# )


from pyspark.sql.functions import count, avg, round, when

top_carriers = df.groupBy("OP_UNIQUE_CARRIER") \
    .agg(
        count("*").alias("total_flights"),
        round(avg("ARR_DELAY"), 2).alias("avg_arr_delay"),
        count(when(col("ARR_DEL15") == 1, 1)).alias("total_delayed")
    ) \
    .orderBy("total_flights", ascending=False) \
    .limit(5)

top_carriers.show()

+-----------------+-------------+-------------+-------------+
|OP_UNIQUE_CARRIER|total_flights|avg_arr_delay|total_delayed|
+-----------------+-------------+-------------+-------------+
|               WN|       345868|          4.8|        71490|
|               AA|       234475|        13.64|        58234|
|               DL|       228270|         0.82|        35437|
|               UA|       180618|         4.04|        32815|
|               OO|       166380|         7.23|        30987|
+-----------------+-------------+-------------+-------------+

